# Inference-time scaling with Qwen3-0.6B-Base

Same model, same chocolate problem — spend more compute at **decode time** and see if answers improve.

**Strategies we demo:**

1. **Chain of Thought (CoT)** — one longer trace: “explain step by step”
2. **Sampling (top-k + top-p)** — stochastic decode for diversity
3. **Self-consistency** — sample several paths, **majority vote**
4. **Best-of-N** — sample several paths, **pick one** with a scorer
5. **Self-refinement** — draft → score / critique → revise (rule-based, length, avg log-prob, LLM-as-judge)

| Step | What we do |
| ---- | ---------- |
| 0 | Load `Qwen/Qwen3-0.6B-Base` |
| 1 | Direct ask (baseline) |
| 2 | CoT prompt |
| 3 | Sampling with top-k + top-p |
| 4 | Self-consistency (parallel + vote) |
| 5 | Best-of-N (sample + pick) |
| 6 | Self-refinement scorers + revise loop |

Run cells top to bottom. Copy useful prompts / outputs into `src/content/concepts/reasoning-models/inference-time-scaling.md`.

> Base models complete text (not chat). Tiny models are noisy — variance across runs is expected and is exactly why voting / scoring / refinement help.


## 0. Setup

Use the project venv if you have it:

```bash
source .venv-qwen/bin/activate
python -m ipykernel install --user --name qwen-demo --display-name "Qwen demo"
```

Then pick kernel **Qwen demo** in this notebook.

Or install once:

```bash
pip install "torch" "transformers>=4.51" accelerate ipykernel
```

In [11]:
from collections import Counter
import re

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("torch", torch.__version__)
print("device will be:", "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))

torch 2.14.0+cpu
device will be: cpu


In [12]:
MODEL_NAME = "Qwen/Qwen3-0.6B-Base"

PROBLEM = (
    "Ram has 3 chocolates. Sam has 5 chocolates. "
    "They put them in one basket and each eats 1. "
    "How many chocolates are left?"
)

print("Loading", MODEL_NAME, "(first run downloads ~1GB)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
)
model.eval()
print("Ready on", model.device)

Loading Qwen/Qwen3-0.6B-Base (first run downloads ~1GB)...


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 3322.21it/s]


Ready on cpu


### Helpers

Keep these small — easy to paste into the MD page later.

In [13]:
def generate(
    prompt: str,
    max_new_tokens: int = 80,
    do_sample: bool = False,
    temperature: float = 0.8,
    top_k: int | None = None,
    top_p: float | None = None,
    num_return_sequences: int = 1,
):
    """Generate one or more completions for a prompt.

    When do_sample=True, pass top_k / top_p to shape the candidate set:
    - top_k: keep only the k highest-probability next tokens
    - top_p: nucleus — keep the smallest set whose probs sum to >= p
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        num_return_sequences=num_return_sequences,
        pad_token_id=tokenizer.eos_token_id,
    )
    if do_sample:
        gen_kwargs["temperature"] = temperature
        if top_k is not None:
            gen_kwargs["top_k"] = top_k
        if top_p is not None:
            gen_kwargs["top_p"] = top_p
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)
    texts = []
    for out in outputs:
        full = tokenizer.decode(out, skip_special_tokens=True)
        texts.append(full[len(prompt) :].strip())
    return texts


def generate_scored(
    prompt: str,
    max_new_tokens: int = 120,
    do_sample: bool = True,
    temperature: float = 0.8,
    top_k: int | None = 40,
    top_p: float | None = 0.9,
    num_return_sequences: int = 1,
):
    """Like generate, but also return mean log-prob of generated tokens (per sequence)."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        num_return_sequences=num_return_sequences,
        pad_token_id=tokenizer.eos_token_id,
        return_dict_in_generate=True,
        output_scores=True,
    )
    if do_sample:
        gen_kwargs["temperature"] = temperature
        if top_k is not None:
            gen_kwargs["top_k"] = top_k
        if top_p is not None:
            gen_kwargs["top_p"] = top_p
    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)

    results = []
    for i, seq in enumerate(out.sequences):
        full = tokenizer.decode(seq, skip_special_tokens=True)
        text = full[len(prompt) :].strip()
        gen_ids = seq[prompt_len:]
        token_lps = []
        for t, step_logits in enumerate(out.scores):
            if t >= len(gen_ids):
                break
            tid = int(gen_ids[t].item())
            if tid == tokenizer.eos_token_id:
                break
            lp = torch.log_softmax(step_logits[i], dim=-1)[tid].item()
            token_lps.append(lp)
        avg_lp = sum(token_lps) / max(len(token_lps), 1)
        results.append((text, avg_lp))
    return results


def extract_answer(text: str):
    """Prefer 'Answer: N', else last number in the text."""
    m = re.search(r"Answer\s*:\s*(\d+)", text, re.I)
    if m:
        return m.group(1)
    nums = re.findall(r"\b(\d+)\b", text)
    return nums[-1] if nums else None


def show(title: str, prompt: str, completion: str):
    print("=" * 60)
    print(title)
    print("=" * 60)
    print("PROMPT:\n")
    print(prompt)
    print("\nCOMPLETION:\n")
    print(completion)
    print("\nPARSED ANSWER:", extract_answer(completion))
    print()


def score_length(text: str) -> float:
    """Toy scorer — longer is not always better."""
    return float(len(text))


def score_rule_based(text: str) -> float:
    """Heuristic verifier — no ground truth. Rewards structure in the trace."""
    ans = extract_answer(text)
    if ans is None:
        return -100.0
    score = 1.0
    if re.search(r"3\s*\+\s*5|5\s*\+\s*3", text):
        score += 2.0
    if re.search(r"(?:total|together).{0,20}8|3\s*\+\s*5\s*=\s*8", text, re.I):
        score += 2.0
    if re.search(r"8\s*-\s*2|1\s*\+\s*1\s*=\s*2", text):
        score += 2.0
    if re.search(r"8\s*-\s*2\s*=\s*6", text):
        score += 3.0
    if re.search(r"Answer\s*:", text, re.I):
        score += 0.5
    return score


## Step 1 — Direct ask (baseline)

No reasoning instruction. Greedy decode (`do_sample=False`).

**Expectation:** often fast and sometimes wrong — no intermediate checks.

In [14]:
prompt_direct = f"Question: {PROBLEM}\nAnswer:"

out_direct = generate(prompt_direct, max_new_tokens=40, do_sample=False)[0]
show("Step 1 — Direct", prompt_direct, out_direct)

Step 1 — Direct
PROMPT:

Question: Ram has 3 chocolates. Sam has 5 chocolates. They put them in one basket and each eats 1. How many chocolates are left?
Answer:

COMPLETION:

Sam has 5 chocolates and Ram has 3 chocolates. They put them in one basket and each eats 1. So, 5 - 1 = 4 chocolates are left. The answer

PARSED ANSWER: 4



## Step 2 — Chain of Thought (CoT)

Same model. Add **“Explain step by step”** and seed `Step 1:` so the base model continues a trace.

**Expectation:** longer completion, checkable arithmetic, often better answer.

In [15]:
prompt_cot = (
    f"Question: {PROBLEM}\n"
    "Explain step by step and think, then write Answer: <number>.\n"
    "Step 1:"
)

out_cot = generate(prompt_cot, max_new_tokens=120, do_sample=False)[0]
show("Step 2 — CoT", prompt_cot, out_cot)

Step 2 — CoT
PROMPT:

Question: Ram has 3 chocolates. Sam has 5 chocolates. They put them in one basket and each eats 1. How many chocolates are left?
Explain step by step and think, then write Answer: <number>.
Step 1:

COMPLETION:

Identify the total number of chocolates.
Ram has 3 chocolates and Sam has 5 chocolates, so the total number of chocolates is 3 + 5 = 8.

Step 2: Determine how many chocolates each person eats.
Each person eats 1 chocolate, so the total number of chocolates eaten is 1 + 1 = 2.

Step 3: Calculate the number of chocolates left.
To find the number of chocolates left, subtract the number of chocolates eaten from the total number of chocolates: 8 - 2 = 6.

Answer: There are 6 chocolates left

PARSED ANSWER: 6



## Step 3 — Sampling with top-k + top-p

Same CoT prompt as Step 2, but **sample** instead of greedy decode.

- **top_k=40** — at each step, only the 40 most likely tokens can be chosen  
- **top_p=0.9** — among those (after top-k), keep the smallest nucleus whose probabilities sum to ≥ 0.9  
- **temperature=0.8** — soften the distribution before sampling  

This is not “more correct by itself” — it creates **diverse** traces. Steps 4–5 then aggregate or select among them.

**Expectation:** several different completions / parsed answers from one prompt.

In [16]:
prompt_sample = (
    f"Question: {PROBLEM}\n"
    "Explain step by step and think, then write Answer: <number>.\n"
    "Step 1:"
)

N_SAMPLE = 4
outs_topk_p = generate(
    prompt_sample,
    max_new_tokens=120,
    do_sample=True,
    temperature=0.8,
    top_k=40,
    top_p=0.9,
    num_return_sequences=N_SAMPLE,
)

print(f"Sampling config: temperature=0.8, top_k=40, top_p=0.9, N={N_SAMPLE}\n")
for i, text in enumerate(outs_topk_p, 1):
    ans = extract_answer(text)
    print(f"--- sample {i} → {ans} ---")
    print(text)
    print()

sample_answers = [extract_answer(t) for t in outs_topk_p]
print("Parsed answers:", sample_answers)
print("Unique answers:", sorted({a for a in sample_answers if a}))


Sampling config: temperature=0.8, top_k=40, top_p=0.9, N=4

--- sample 1 → 2 ---
Identify the total number of chocolates Ram and Sam have together.
- Ram has 3 chocolates.
- Sam has 5 chocolates.
- Total chocolates = 3 + 5 = 8 chocolates.

Step 2: Determine how many chocolates each person eats.
- Each person eats 1 chocolate.
- Since there are two people (Ram and Sam), they eat a total of 1 + 1 = 2 chocolates.

Step 3: Calculate the number of chocolates left after each person eats.
- Total chocolates initially = 8.
- Chocolates eaten = 2.
- Ch

--- sample 2 → 6 ---
Add the number of chocolates Ram and Sam have.
Ram has 3 chocolates.
Sam has 5 chocolates.
Total number of chocolates = 3 + 5 = 8

Step 2: Subtract the number of chocolates each eats.
Each of them eats 1 chocolate.
Total number of chocolates eaten = 1 + 1 = 2

Step 3: Subtract the number of chocolates eaten from the total number of chocolates.
Total number of chocolates left = 8 - 2 = 6

Answer: There are 6 chocolates left.


## Step 4 — Self-consistency (parallel strategy)

Sample **N** CoT completions with the same **top-k + top-p** knobs as Step 3, parse each answer, take the **majority vote**.

This is the usual “parallel + sampling” combo people mean by self-consistency (Wang et al.).

**Expectation:** single bad samples get outvoted when the model is usually right.

In [17]:
N = 5  # raise to 8–16 on GPU if you want a clearer vote

prompt_sc = (
    f"Question: {PROBLEM}\n"
    "Explain step by step and think, then write Answer: <number>.\n"
    "Step 1:"
)

samples = generate(
    prompt_sc,
    max_new_tokens=120,
    do_sample=True,
    temperature=0.8,
    top_k=40,
    top_p=0.9,
    num_return_sequences=N,
)

answers = []
for i, text in enumerate(samples, 1):
    ans = extract_answer(text)
    answers.append(ans)
    print(f"--- sample {i} → {ans} ---")
    print(text)
    print()

votes = Counter(a for a in answers if a is not None)
majority = votes.most_common(1)[0][0] if votes else None

print("Votes:", dict(votes))
print("Majority answer:", majority)

--- sample 1 → 2 ---
Determine the total number of chocolates. Ram has 3 chocolates and Sam has 5 chocolates. To find the total, add the number of chocolates each has: 3 + 5 = 8.
Step 2: Determine how many chocolates are eaten. Each person eats 1 chocolate. Since there are 2 people (Ram and Sam), they eat a total of 1 + 1 = 2 chocolates.
Step 3: Subtract the number of chocolates eaten from the total number of chocolates. There are 8 chocolates initially and 2 chocolates have been eaten. To find the remaining

--- sample 2 → 2 ---
Identify the total number of chocolates.
Ram has 3 chocolates, and Sam has 5 chocolates. So, the total number of chocolates is 3 + 5 = 8.

Step 2: Determine how many chocolates each person eats.
Ram eats 1 chocolate, and Sam eats 1 chocolate. So, they eat a total of 1 + 1 = 2 chocolates.

Step 3: Subtract the number of chocolates eaten from the total number of chocolates.
Now, subtract the 2 chocolates that were eaten from the total of 8 chocolates: 8 - 2 =

-

## Step 5 — Best-of-N (sampling + select)

Same samples as Step 4, but instead of voting we **pick one** completion with a scorer.

Here the scorer is intentionally naive: **longest completion** (more steps ≈ more “effort”).  
Real systems use a process/outcome reward model or a verifier.

**Expectation:** you get one detailed trace to show; quality depends on the scorer.

In [10]:
# Reuse `samples` from Step 4. Re-run Step 4 first if this cell errors.

def score_length(text: str) -> float:
    """Toy scorer — replace with a reward model in real setups."""
    return float(len(text))


ranked = sorted(
    ((score_length(t), extract_answer(t), t) for t in samples),
    key=lambda x: x[0],
    reverse=True,
)

print("Ranked by toy score (length):")
for rank, (score, ans, text) in enumerate(ranked, 1):
    print(f"  #{rank}  score={score:.0f}  answer={ans}")

best_score, best_ans, best_text = ranked[0]
print("\n=== Best-of-N pick ===")
print(best_text)
print("\nSelected answer:", best_ans)

Ranked by toy score (length):
  #1  score=502  answer=3
  #2  score=489  answer=6
  #3  score=482  answer=4
  #4  score=451  answer=6
  #5  score=448  answer=7

=== Best-of-N pick ===
Determine the initial number of chocolates
Ram has 3 chocolates and Sam has 5 chocolates. To find the total number of chocolates, add the number of chocolates each person has:
3 (Ram's chocolates) + 5 (Sam's chocolates) = 8 chocolates.

Step 2: Determine the number of chocolates eaten
Both Ram and Sam each eat 1 chocolate. To find the total number of chocolates eaten, add the number of chocolates eaten by each person:
1 (Ram's chocolate) + 1 (Sam's chocolate) = 2 chocolates.

Step 3: Calculate the

Selected answer: 3


## Step 6 — Self-refinement

Parallel methods (Steps 4–5) draw many answers at once. **Self-refinement** is sequential:

1. **Draft** an answer  
2. **Score / critique** it (or rank candidates)  
3. **Revise** if needed — then optionally loop  

Common scorers / correctors (what we demo):

| Strategy | Idea |
| -------- | ---- |
| Rule-based | Cheap checks (parseable answer, arithmetic patterns in the trace) |
| Length | Prefer longer traces (toy; often wrong — see Step 5) |
| Avg log-prob | Prefer completions the model assigns higher mean token log-prob |
| LLM-as-judge | Same model critiques or picks the best candidate |
| Refine loop | Feed critique back and regenerate a corrected solution |

> Same weights — extra compute is more generate / judge / revise calls.


### 6a — Scorers: rule-based, length, avg log-prob

Reuse CoT sampling. Rank the same candidates three ways and compare picks.


In [ ]:
prompt_refine = (
    f"Question: {PROBLEM}\n"
    "Explain step by step and think, then write Answer: <number>.\n"
    "Step 1:"
)

# Fresh scored samples (text, avg_logprob). Or reuse `samples` from Step 4 if present.
scored = generate_scored(
    prompt_refine,
    max_new_tokens=120,
    do_sample=True,
    temperature=0.8,
    top_k=40,
    top_p=0.9,
    num_return_sequences=5,
)
cand_texts = [t for t, _ in scored]

print("Candidates:")
for i, (text, avg_lp) in enumerate(scored, 1):
    print(
        f"  #{i} ans={extract_answer(text)}  "
        f"avg_logprob={avg_lp:.3f}  len={len(text)}  rule={score_rule_based(text):.1f}"
    )

pick_len = max(cand_texts, key=score_length)
pick_rule = max(cand_texts, key=score_rule_based)
pick_lp = max(scored, key=lambda p: p[1])[0]

print("\nPick by length:       ", extract_answer(pick_len))
print("Pick by rule-based:   ", extract_answer(pick_rule), f"(score={score_rule_based(pick_rule):.1f})")
print("Pick by avg log-prob: ", extract_answer(pick_lp))


### 6b — LLM-as-judge (pick among candidates)

Same model reads short candidate summaries and outputs `Judge: <n>`. Tiny models are unreliable judges — still useful to see the pattern.


In [ ]:
# Use the scored candidates from 6a
summaries = []
for i, text in enumerate(cand_texts, 1):
    ans = extract_answer(text)
    snippet = text.replace("\n", " ")[:180]
    summaries.append(f"Candidate {i}: answer={ans}\n{snippet}")

judge_prompt = (
    f"Question: {PROBLEM}\n"
    "Pick the best candidate. Reply with only: Judge: <number>\n\n"
    + "\n\n".join(summaries)
    + "\n\nJudge:"
)

judge_out = generate(judge_prompt, max_new_tokens=16, do_sample=False)[0]
print("Judge raw:\n", judge_out)

m = re.search(r"Judge\s*:\s*(\d+)", judge_out, re.I) or re.search(r"\b(\d+)\b", judge_out)
judge_idx = int(m.group(1)) if m else None
if judge_idx and 1 <= judge_idx <= len(cand_texts):
    picked = cand_texts[judge_idx - 1]
    print(f"\nLLM-as-judge picked #{judge_idx} → answer={extract_answer(picked)}")
else:
    print("\nCould not parse judge index; fall back to rule-based.")
    picked = pick_rule
    print("Fallback answer:", extract_answer(picked))


### 6c — Refine loop (critique → revise)

Start from a **weak draft** (direct ask), ask the model to critique, then regenerate with that feedback. Optional: stop early if `Verdict: OK`.


In [ ]:
# Weak draft on purpose (baseline-style)
draft_prompt = f"Question: {PROBLEM}\nAnswer:"
draft = generate(draft_prompt, max_new_tokens=40, do_sample=False)[0]
print("DRAFT answer:", extract_answer(draft))
print(draft)
print()

critique_prompt = (
    f"Question: {PROBLEM}\n"
    f"Draft solution:\n{draft}\n\n"
    "Critique the draft briefly. Check the arithmetic.\n"
    "End with exactly one of: Verdict: OK  or  Verdict: REVISE\n"
    "Critique:"
)
critique = generate(critique_prompt, max_new_tokens=80, do_sample=False)[0]
print("CRITIQUE:\n", critique)
print()

need_revise = ("REVISE" in critique.upper()) or (score_rule_based(draft) < 3.0)

if need_revise:
    revise_prompt = (
        f"Question: {PROBLEM}\n"
        f"Previous draft:\n{draft}\n\n"
        f"Feedback:\n{critique}\n\n"
        "Write a corrected solution step by step, then Answer: <number>.\n"
        "Step 1:"
    )
    refined = generate(revise_prompt, max_new_tokens=120, do_sample=False)[0]
    print("REFINED answer:", extract_answer(refined))
    print(refined)
else:
    refined = draft
    print("Kept draft (verdict OK).")


### 6d — Multi-round refine (optional)

Repeat critique → revise for a few rounds, or stop when the rule-based score stops improving.


In [ ]:
def self_refine(problem: str, rounds: int = 2):
    """Draft → critique → revise for a few rounds; keep best by rule-based score."""
    state_prompt = f"Question: {problem}\nAnswer:"
    state = generate(state_prompt, max_new_tokens=40, do_sample=False)[0]
    best = state
    best_score = score_rule_based(state)
    history = [("draft", extract_answer(state), best_score)]

    for r in range(1, rounds + 1):
        critique_prompt = (
            f"Question: {problem}\n"
            f"Draft solution:\n{state}\n\n"
            "Critique briefly. End with Verdict: OK or Verdict: REVISE\n"
            "Critique:"
        )
        critique = generate(critique_prompt, max_new_tokens=60, do_sample=False)[0]
        revise_prompt = (
            f"Question: {problem}\n"
            f"Previous draft:\n{state}\n\n"
            f"Feedback:\n{critique}\n\n"
            "Corrected solution. Explain step by step, then Answer: <number>.\n"
            "Step 1:"
        )
        state = generate(revise_prompt, max_new_tokens=120, do_sample=False)[0]
        s = score_rule_based(state)
        history.append((f"round-{r}", extract_answer(state), s))
        if s >= best_score:
            best, best_score = state, s

    return best, history


final, hist = self_refine(PROBLEM, rounds=2)
print("History (stage, answer, rule_score):")
for row in hist:
    print(" ", row)
print("\nBest answer:", extract_answer(final), f"(rule_score={score_rule_based(final):.1f})")


## Compare (fill after you run)

| Step | Strategy | Parsed answer | Notes |
| ---- | -------- | ------------- | ----- |
| 1 | Direct | ? | |
| 2 | CoT | ? | |
| 3 | top-k + top-p samples | ? | answers: |
| 4 | Self-consistency | ? | votes: |
| 5 | Best-of-N (length) | ? | |
| 6a | Rule / length / avg log-prob picks | ? | |
| 6b | LLM-as-judge | ? | |
| 6c–d | Self-refine | ? | |

Correct answer: **6**.

Takeaway: parallel vote/select **or** sequential critique→revise — both spend decode compute without changing weights.


## Optional: one-shot compare helper

Re-run core strategies (skip multi-round refine — slow on CPU).


In [ ]:
def run_all(n_samples: int = 5):
    direct = generate(prompt_direct, max_new_tokens=40, do_sample=False)[0]
    cot = generate(prompt_cot, max_new_tokens=120, do_sample=False)[0]
    scored = generate_scored(
        prompt_cot,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.8,
        top_k=40,
        top_p=0.9,
        num_return_sequences=n_samples,
    )
    texts = [t for t, _ in scored]
    answers = [extract_answer(t) for t in texts]
    vote = Counter(a for a in answers if a).most_common(1)
    by_len = max(texts, key=len)
    by_rule = max(texts, key=score_rule_based)
    by_lp = max(scored, key=lambda p: p[1])[0]

    print("Direct:            ", extract_answer(direct))
    print("CoT:               ", extract_answer(cot))
    print("Self-consistency:  ", vote[0][0] if vote else None, "votes=", dict(Counter(answers)))
    print("Best-of-N length:  ", extract_answer(by_len))
    print("Best-of-N rule:    ", extract_answer(by_rule))
    print("Best-of-N avg_lp:  ", extract_answer(by_lp))


# Uncomment when you want a fresh summary (slow on CPU):
# run_all(5)
